## 本地缓存 + 记忆 + 提示词模板

In [1]:
import os
import json
from llama_index.core import Document, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core import PromptTemplate, ChatPromptTemplate

from core.config import load_config
from core.models import get_zhipu_embedding, get_zhipu_llm

## 1.构建模型

In [2]:
def load_documents_from_folder(folder_path: str):
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if not os.path.isfile(file_path):
            continue
        ext = os.path.splitext(filename)[1].lower()
        try:
            if ext == ".txt":
                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()
                print(f"读取文本文件: {filename}，字符数: {len(text)}")
                documents.append(Document(text=text, metadata={"file_name": filename}))
            elif ext == ".docx":
                from docx import Document as DocxDocument
                doc = DocxDocument(file_path)
                paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
                table_texts = []
                for table in doc.tables:
                    for row in table.rows:
                        for cell in row.cells:
                            if cell.text.strip():
                                table_texts.append(cell.text.strip())
                full_text = "\n".join(paragraphs + table_texts)
                print(f"读取 Word 文件: {filename}，字符数: {len(full_text)}")
                documents.append(Document(text=full_text, metadata={"file_name": filename}))
            else:
                print(f"⏭忽略不支持的文件: {filename}")
        except Exception as e:
            print(f"读取文件 {filename} 时出错: {e}")
            raise
    print(f"总共加载了 {len(documents)} 个文档。")
    return documents

## 2.构建加载索引

In [3]:
def build_or_load_index(vector_path, docs_path, embed_model, chunk_size=512, chunk_overlap=50):
    os.makedirs(vector_path, exist_ok=True)
    if not os.listdir(vector_path):
        print("向量库为空，开始构建...")
        docs = load_documents_from_folder(docs_path)
        if not docs:
            raise RuntimeError("未找到任何可读文档，请检查 docs/ 目录。")
        splitter = SentenceSplitter(
            separator="\n\n",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        nodes = splitter.get_nodes_from_documents(docs)
        index = VectorStoreIndex(nodes, embed_model=embed_model)
        index.storage_context.persist(vector_path)
        print("向量存储完成，已保存到", vector_path)
    else:
        print("检测到已有向量库，直接加载...")
        storage_context = StorageContext.from_defaults(persist_dir=vector_path)
        index = load_index_from_storage(storage_context, embed_model=embed_model)
        print("向量库加载完成")
    return index

## 3.初始化模型和索引

In [4]:
config = load_config("config.yaml")
embed_model = get_zhipu_embedding(config)
zhipu_llm = get_zhipu_llm(config)

docs_path = "./docs/"
vector_path = "./vector_store/"

index = build_or_load_index(vector_path, docs_path, embed_model)
vector = index.as_retriever(similarity_top_k=5)

检测到已有向量库，直接加载...
向量库加载完成


### **4. 增加记忆**
1. **导入记忆模块** 
2. **配置记忆** 
3. **测试记忆**

In [5]:
# 构建记忆库
memory_buffer = ChatMemoryBuffer(token_limit=2048)

# 配置记忆库
chat_engine = CondensePlusContextChatEngine.from_defaults(
    vector,
    memory=memory_buffer,
    llm=zhipu_llm
)

In [6]:
chat_engine.chat("根据上下文回答问题，简明扼要。小娟姓什么？")

AgentChatResponse(response='小娟姓张。', sources=[ToolOutput(blocks=[TextBlock(block_type='text', text="[NodeWithScore(node=TextNode(id_='bac865d9-cdd8-4c63-87d8-5c46be5610f0', embedding=None, metadata={'file_name': 'test.txt'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='21af5350-0999-432b-8119-ae06956c3953', node_type='4', metadata={'file_name': 'test.txt'}, hash='fe22acfae8a6d7cb88d472755e6c45c81abd136eb840ef55abc11373720ef2eb')}, metadata_template='{key}: {value}', metadata_separator='\\n', text='张老师有一个学生叫张小娟。小娟是计算机科学专业的学生，今年大三。她的成绩非常优秀，尤其是在人工智能课程上表现突出。小娟喜欢编程，经常参加各种编程比赛。她的梦想是成为一名优秀的AI工程师。', mimetype='text/plain', start_char_idx=0, end_char_idx=93, text_template='{metadata_str}\\n\\n{content}'), score=0.4628376721006295), NodeWithScore(node=TextNode(id_='615711c1-29d9-41f2-b5fe-afc297e612e2', embedding=None, metadata={'file_name': 'new.docx'}, excluded_embed_metadata_keys=[], excluded_llm_metadata

In [7]:
chat_engine.chat("我的上一个问题是什么？")

AgentChatResponse(response='小娟姓什么？', sources=[ToolOutput(blocks=[TextBlock(block_type='text', text="[NodeWithScore(node=TextNode(id_='bac865d9-cdd8-4c63-87d8-5c46be5610f0', embedding=None, metadata={'file_name': 'test.txt'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='21af5350-0999-432b-8119-ae06956c3953', node_type='4', metadata={'file_name': 'test.txt'}, hash='fe22acfae8a6d7cb88d472755e6c45c81abd136eb840ef55abc11373720ef2eb')}, metadata_template='{key}: {value}', metadata_separator='\\n', text='张老师有一个学生叫张小娟。小娟是计算机科学专业的学生，今年大三。她的成绩非常优秀，尤其是在人工智能课程上表现突出。小娟喜欢编程，经常参加各种编程比赛。她的梦想是成为一名优秀的AI工程师。', mimetype='text/plain', start_char_idx=0, end_char_idx=93, text_template='{metadata_str}\\n\\n{content}'), score=0.40653091045671613), NodeWithScore(node=TextNode(id_='615711c1-29d9-41f2-b5fe-afc297e612e2', embedding=None, metadata={'file_name': 'new.docx'}, excluded_embed_metadata_keys=[], excluded_llm_metada

### **5. 提示词模板**

In [8]:
content="小娟姓什么？"
message_template = [
    ChatMessage(role=MessageRole.SYSTEM,content="你是一个经验丰富体育专家，擅长足球领域、逻辑清晰、回答问题严谨而简洁。"),
    ChatMessage(role=MessageRole.SYSTEM,content="回答问题，最终答案在前，逻辑推理在后，用文中的原文来佐证结果"),
    ChatMessage(role=MessageRole.USER,content=content)
    ]

chat_template = ChatPromptTemplate(message_template)
prompt= chat_template.format()
prompt

'system: 你是一个经验丰富体育专家，擅长足球领域、逻辑清晰、回答问题严谨而简洁。\nsystem: 回答问题，最终答案在前，逻辑推理在后，用文中的原文来佐证结果\nuser: 小娟姓什么？\nassistant: '

In [9]:
chat_engine.chat(prompt)

AgentChatResponse(response='小娟姓张。\n\n根据提供的文档，原文中明确提到：“张老师有一个学生叫张小娟。” 因此，小娟姓张。', sources=[ToolOutput(blocks=[TextBlock(block_type='text', text="[NodeWithScore(node=TextNode(id_='bac865d9-cdd8-4c63-87d8-5c46be5610f0', embedding=None, metadata={'file_name': 'test.txt'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='21af5350-0999-432b-8119-ae06956c3953', node_type='4', metadata={'file_name': 'test.txt'}, hash='fe22acfae8a6d7cb88d472755e6c45c81abd136eb840ef55abc11373720ef2eb')}, metadata_template='{key}: {value}', metadata_separator='\\n', text='张老师有一个学生叫张小娟。小娟是计算机科学专业的学生，今年大三。她的成绩非常优秀，尤其是在人工智能课程上表现突出。小娟喜欢编程，经常参加各种编程比赛。她的梦想是成为一名优秀的AI工程师。', mimetype='text/plain', start_char_idx=0, end_char_idx=93, text_template='{metadata_str}\\n\\n{content}'), score=0.40653091045671613), NodeWithScore(node=TextNode(id_='615711c1-29d9-41f2-b5fe-afc297e612e2', embedding=None, metadata={'file_name': 'new.docx'}, excluded_

### **记忆重置**

In [ ]:
memory_buffer

memory_buffer.reset()
memory_buffer